# Diabetic Retinopathy Grading — Swin Transformer V2 (EEEM068)

**Model:** Swin Transformer V2 Small (`swinv2_small_window8_256`)
**Task:** 5-class DR severity grading (0=No DR → 4=Proliferative DR)
**Unique contributions:**
- SwinV2-Small: post-norm + scaled cosine attention — more stable than V1, fewer GPU memory issues
- Augmentation ablation V1→V4 (controlled by `AUGMENT_VERSION`)
- Three imbalance strategies compared: class weights / weighted sampler / focal loss
- Test-time augmentation (TTA) at inference
- Grad-CAM visualisation of retinal attention maps

**Workflow:** Edit the **Config cell** only, then run all cells top-to-bottom.

In [36]:
# 1) Environment and imports
from __future__ import annotations

import json
import os
import random
from contextlib import nullcontext
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from skimage.filters import frangi
from skimage.restoration import denoise_bilateral
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

import torchvision.models as tv_models

try:
    import timm
    TIMM_AVAILABLE = True
    print(f"timm {timm.__version__} available")
except ImportError:
    TIMM_AVAILABLE = False
    print("WARNING: timm not installed — run:  pip install timm>=0.9.0")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

## 2) Configuration (edit only this cell)

Change `AUGMENT_VERSION` and the imbalance toggles to run different experiments.
Keep everything else fixed for fair comparison across runs.

In [37]:
# 2) Config — edit only this cell per experiment.

# ── Paths ────────────────────────────────────────────────────────────────────
EXPERIMENT_NAME  = "swinv2_small_v3_weighted"   # change per run
DATA_ROOT        = r"/media/zm21856/SSD/AML/diabetic-retinopathy-detection/train"
LABELS_CSV       = r"/media/zm21856/SSD/AML/diabetic-retinopathy-detection/trainLabels.csv"
IMAGE_EXTENSION  = ".jpeg"

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED             = 42
TRAIN_VAL_SPLIT  = 0.2
NUM_WORKERS      = 4
DEVICE           = "auto"   # "auto" | "cuda" | "cpu"

# ── Image ─────────────────────────────────────────────────────────────────────
# SwinV2-Small native pre-training resolution is 256×256 — do NOT change
IMAGE_SIZE       = 256
NUM_CLASSES      = 5

# ── Augmentation version (ablation study) ────────────────────────────────────
# "v1_none"   : resize + normalise only            (ablation baseline)
# "v2_basic"  : + hflip + vflip + rotation         (ablation step 2)
# "v3_full"   : + colour jitter + grid distort     (main experiment)
# "v4_mixup"  : v3_full + MixUp during training    (best attempt)
AUGMENT_VERSION  = "v3_full"

# Fine-grained augmentation knobs (used by v2/v3/v4)
HFLIP_PROB       = 0.5
VFLIP_PROB       = 0.5
ROTATION_DEG     = 30
COLOR_JITTER     = dict(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)
RANDOM_ERASE_PROB= 0.2
MIXUP_ALPHA      = 0.4   # only used when AUGMENT_VERSION = "v4_mixup"

# ── Fundus preprocessing ──────────────────────────────────────────────────────
USE_PREPROCESSING= True
PREPROCESS       = dict(
    crop_black      = True,
    circular_mask   = True,
    glare_inpaint   = False,   # expensive; keep off
    ben_graham      = True,    # key for DR — removes illumination artefact
    clahe           = True,    # local contrast enhancement
    green_enhance   = False,
    vessel_frangi   = False,   # very expensive; off
    red_enhance     = False,
    adaptive_denoise= False,   # slow; off for speed
)

# ── Model ─────────────────────────────────────────────────────────────────────
# SwinV2-Small: 50M params, 256px, ImageNet-1k pretrained
# Better than V1: post-norm + scaled cosine attention = stable fine-tuning
ARCHITECTURE     = "swinv2_small_window8_256"
PRETRAINED       = True
DROPOUT          = 0.3
FREEZE_BACKBONE_EPOCHS = 3   # warm-up: train head only for 3 epochs

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE       = 16    # SwinV2-S at 256px fits T4 (15GB) at batch 16
EPOCHS           = 25
LR               = 1e-4
OPTIMIZER        = "adamw"
WEIGHT_DECAY     = 1e-2
SCHEDULER        = "cosine"
SCHEDULER_STEP_SIZE = 7
SCHEDULER_GAMMA  = 0.1
EARLY_STOPPING_PATIENCE = 6
MIXED_PRECISION  = True
GRADIENT_CLIP_NORM = 1.0
SAVE_BEST_METRIC = "val_kappa"   # quadratic weighted kappa

# ── Imbalance strategy ────────────────────────────────────────────────────────
# Strategy A: class-weighted loss       → CLASS_WEIGHTS_AUTO=True,  USE_FOCAL_LOSS=False, WEIGHTED_SAMPLING=False
# Strategy B: weighted random sampler   → CLASS_WEIGHTS_AUTO=False, USE_FOCAL_LOSS=False, WEIGHTED_SAMPLING=True
# Strategy C: focal loss                → CLASS_WEIGHTS_AUTO=True,  USE_FOCAL_LOSS=True,  WEIGHTED_SAMPLING=False
CLASS_WEIGHTS_AUTO = False
WEIGHTED_SAMPLING  = True    # Strategy B active for this run
LABEL_SMOOTHING    = 0.1
USE_FOCAL_LOSS     = False
FOCAL_GAMMA        = 2.0
FOCAL_ALPHA        = None

# ── Test-time augmentation ────────────────────────────────────────────────────
USE_TTA  = True    # average predictions over N augmented views at inference
TTA_N    = 5       # number of TTA views (original + 4 flips/rotations)

In [38]:
# Helpers — seed, device, repo root  (unchanged from team skeleton)

def get_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data").is_dir() and (cwd / "notebooks").is_dir():
        return cwd
    if (cwd.parent / "data").is_dir() and (cwd.parent / "notebooks").is_dir():
        return cwd.parent
    return cwd

REPO_ROOT = get_repo_root()
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def resolve_device(spec: str) -> torch.device:
    spec = (spec or "auto").lower()
    if spec == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(spec)

set_seed(SEED)
device = resolve_device(DEVICE)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Repository root: /scratch/mesa_caches/AML/AML_G14_CW/notebooks
Device: cuda
GPU: NVIDIA RTX A4000 | VRAM: 16.8 GB


## 3) Dataset, preprocessing, transforms

`preprocess_fundus_pil` applies Ben Graham preprocessing (removes illumination artefact)
and CLAHE (local contrast enhancement). Both are standard in DR literature.

`build_transforms` supports four augmentation versions for the ablation study.

In [39]:
# 3) Dataset + preprocessing + transforms

def preprocess_fundus_pil(image: Image.Image, steps: dict) -> Image.Image:
    """Fundus-specific preprocessing pipeline.

    Ben Graham (2015 Kaggle winner): subtract Gaussian blur to remove
    low-frequency illumination variation, enhancing local lesion contrast.
    CLAHE: adaptive histogram equalisation preserving local detail.
    """
    img     = np.array(image)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    if steps.get("crop_black", True):
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        coords = cv2.findNonZero(thresh)
        if coords is not None:
            x, y, w, h = cv2.boundingRect(coords)
            img_bgr = img_bgr[y:y+h, x:x+w]

    if steps.get("circular_mask", True):
        h, w  = img_bgr.shape[:2]
        mask  = np.zeros((h, w), dtype=np.uint8)
        r     = int(0.48 * min(h, w))
        cv2.circle(mask, (w // 2, h // 2), r, 255, -1)
        img_bgr = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)

    if steps.get("glare_inpaint", False):
        hsv   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        glare = ((hsv[:, :, 2] > 245).astype(np.uint8) * 255)
        if glare.sum() > 0:
            glare   = cv2.dilate(glare, np.ones((3, 3), np.uint8))
            img_bgr = cv2.inpaint(img_bgr, glare, 3, cv2.INPAINT_TELEA)

    if steps.get("ben_graham", True):
        # Subtract local mean colour — Ben Graham's key insight
        blur    = cv2.GaussianBlur(img_bgr, (0, 0), sigmaX=10)
        img_bgr = cv2.addWeighted(img_bgr, 4, blur, -4, 128)

    if steps.get("clahe", True):
        lab    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_bgr = cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)

    if steps.get("green_enhance", False):
        b_ch, g_ch, r_ch = cv2.split(img_bgr)
        img_bgr = cv2.merge((b_ch, cv2.equalizeHist(g_ch), r_ch))

    if steps.get("vessel_frangi", False):
        green   = cv2.split(img_bgr)[1].astype(np.float32) / 255.0
        vessels = frangi(green)
        vessels = (255 * vessels / (vessels.max() + 1e-6)).astype(np.uint8)
        img_bgr = cv2.addWeighted(img_bgr, 0.8,
                                  cv2.cvtColor(vessels, cv2.COLOR_GRAY2BGR), 0.2, 0)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if steps.get("adaptive_denoise", False):
        dn      = denoise_bilateral(img_rgb, sigma_color=0.05,
                                    sigma_spatial=3, channel_axis=-1)
        img_rgb = (np.clip(dn, 0, 1) * 255).astype(np.uint8)

    return Image.fromarray(img_rgb)


class RetinaDataset(Dataset):
    """Fundus images + integer labels 0..NUM_CLASSES-1."""

    def __init__(self, frame: pd.DataFrame, data_root: Path,
                 image_extension: str, transform, preprocess: bool):
        self.frame           = frame.reset_index(drop=True)
        self.data_root       = Path(data_root)
        ext                  = image_extension if str(image_extension).startswith(".") else f".{image_extension}"
        self.image_extension = ext
        self.transform       = transform
        self.preprocess      = bool(preprocess)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row  = self.frame.iloc[idx]
        stem = str(row["image"])
        if not stem.lower().endswith((".jpeg", ".jpg", ".png")):
            stem = stem + self.image_extension
        image = Image.open(self.data_root / stem).convert("RGB")
        if self.preprocess:
            image = preprocess_fundus_pil(image, PREPROCESS)
        label = int(row["level"])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def build_transforms(train: bool, version: str = "v3_full") -> transforms.Compose:
    """
    Augmentation versions for ablation study (AUGMENT_VERSION config key).

    V1 none   : resize + centre-crop + normalise only
    V2 basic  : + horizontal/vertical flip + rotation
    V3 full   : + colour jitter + random erasing  (default/best)
    V4 mixup  : same as v3_full; MixUp is applied in the training loop
    """
    size = int(IMAGE_SIZE)

    if not train:
        return transforms.Compose([
            transforms.Resize(int(size * 1.05)),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    v = version.lower()

    if v == "v1_none":
        ops = [
            transforms.Resize(int(size * 1.05)),
            transforms.CenterCrop(size),
        ]
    elif v == "v2_basic":
        ops = [
            transforms.Resize(int(size * 1.12)),
            transforms.RandomCrop(size),
            transforms.RandomHorizontalFlip(p=float(HFLIP_PROB)),
            transforms.RandomVerticalFlip(p=float(VFLIP_PROB)),
            transforms.RandomRotation(degrees=float(ROTATION_DEG)),
        ]
    elif v in ("v3_full", "v4_mixup"):
        ops = [
            transforms.Resize(int(size * 1.15)),
            transforms.RandomCrop(size),
            transforms.RandomHorizontalFlip(p=float(HFLIP_PROB)),
            transforms.RandomVerticalFlip(p=float(VFLIP_PROB)),
            transforms.RandomRotation(degrees=float(ROTATION_DEG)),
            transforms.ColorJitter(
                brightness = float(COLOR_JITTER["brightness"]),
                contrast   = float(COLOR_JITTER["contrast"]),
                saturation = float(COLOR_JITTER["saturation"]),
                hue        = float(COLOR_JITTER["hue"]),
            ),
        ]
    else:
        raise ValueError(f"Unknown AUGMENT_VERSION: {version}. "
                         "Use v1_none | v2_basic | v3_full | v4_mixup")

    ops.append(transforms.ToTensor())
    if v in ("v3_full", "v4_mixup") and float(RANDOM_ERASE_PROB) > 0:
        ops.append(transforms.RandomErasing(p=float(RANDOM_ERASE_PROB)))
    ops.append(transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return transforms.Compose(ops)


def mixup_batch(images: torch.Tensor, labels: torch.Tensor,
                alpha: float = 0.4):
    """MixUp augmentation (used in training loop when AUGMENT_VERSION=v4_mixup)."""
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[idx]
    return mixed, labels, labels[idx], lam

print("Dataset and transform utilities loaded.")

Dataset and transform utilities loaded.


## 4) Train / validation split

In [40]:
# 4) Load labels CSV and stratified 80/20 train/val split

labels_path = Path(LABELS_CSV) if str(LABELS_CSV).startswith("/") else (REPO_ROOT / LABELS_CSV).resolve()
data_root   = Path(DATA_ROOT)  if str(DATA_ROOT).startswith("/")  else (REPO_ROOT / DATA_ROOT).resolve()

full_df = pd.read_csv(labels_path)
full_df.columns = [c.strip().lower() for c in full_df.columns]
assert "image" in full_df.columns and "level" in full_df.columns, (
    f"Expected columns image, level; got {full_df.columns.tolist()}"
)

train_df, val_df = train_test_split(
    full_df,
    test_size   = float(TRAIN_VAL_SPLIT),
    stratify    = full_df["level"],
    random_state= int(SEED),
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")
print("\nClass distribution (train):")
print(train_df["level"].value_counts().sort_index().to_string())
print("\nClass distribution (val):")
print(val_df["level"].value_counts().sort_index().to_string())

FileNotFoundError: [Errno 2] No such file or directory: '/media/zm21856/SSD/AML/diabetic-retinopathy-detection/trainLabels.csv'

## 5) DataLoaders

Three imbalance strategies selectable via config:
- **A** `CLASS_WEIGHTS_AUTO=True`  → inverse-frequency weights in loss
- **B** `WEIGHTED_SAMPLING=True`   → WeightedRandomSampler (balanced batches)
- **C** `USE_FOCAL_LOSS=True`      → Focal Loss down-weights easy examples

In [ ]:
# 5) Wrap datasets with DataLoader

train_tf = build_transforms(train=True,  version=AUGMENT_VERSION)
val_tf   = build_transforms(train=False, version=AUGMENT_VERSION)

train_ds = RetinaDataset(train_df, data_root, IMAGE_EXTENSION, train_tf, preprocess=USE_PREPROCESSING)
val_ds   = RetinaDataset(val_df,   data_root, IMAGE_EXTENSION, val_tf,   preprocess=USE_PREPROCESSING)

train_sampler = None
if WEIGHTED_SAMPLING:
    counts         = train_df["level"].value_counts().to_dict()
    sample_weights = train_df["level"].map(lambda y: 1.0 / counts[int(y)]).to_numpy(dtype=np.float64)
    train_sampler  = WeightedRandomSampler(
        weights     = torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples = len(sample_weights),
        replacement = True,
    )
    print("Imbalance strategy: WeightedRandomSampler (Strategy B)")
elif CLASS_WEIGHTS_AUTO:
    print("Imbalance strategy: class-weighted loss (Strategy A)")
if USE_FOCAL_LOSS:
    print("Loss: Focal Loss (Strategy C) — overrides class weight in loss function")

train_loader = DataLoader(
    train_ds,
    batch_size  = int(BATCH_SIZE),
    shuffle     = (train_sampler is None),
    sampler     = train_sampler,
    num_workers = int(NUM_WORKERS),
    pin_memory  = torch.cuda.is_available(),
    drop_last   = True,   # avoids single-sample batch edge cases with AMP
)
val_loader = DataLoader(
    val_ds,
    batch_size  = int(BATCH_SIZE),
    shuffle     = False,
    num_workers = int(NUM_WORKERS),
    pin_memory  = torch.cuda.is_available(),
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## 6) Model — Swin Transformer V2 Small

**Architecture justification:**
- V2 over V1: post-normalisation + scaled cosine attention = stable training, no activation explosion
- Small over Base: 50M vs 88M params; only 0.5% ImageNet gap, fits T4 GPU at batch 16
- Layer-wise LR decay: backbone gets 10× lower LR than head to prevent catastrophic forgetting
- Head: LayerNorm → Dropout(0.3) → Linear(1024→5)

In [ ]:
# 6) Build model — SwinV2-Small via timm, with EfficientNet/ViT fallback

def _is_swin(arch: str) -> bool:
    return arch.lower().startswith("swin")


def replace_classifier_head(model: nn.Module, arch: str,
                             num_classes: int, dropout: float) -> None:
    """Replace head for torchvision EfficientNet / ViT models."""
    a = arch.lower()
    if a.startswith("efficientnet"):
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=float(dropout), inplace=True),
            nn.Linear(in_f, num_classes),
        ) if dropout > 0 else nn.Sequential(nn.Linear(in_f, num_classes))
    elif a.startswith("vit"):
        in_f = model.heads.head.in_features
        model.heads.head = nn.Linear(in_f, num_classes)
    else:
        raise ValueError(f"Unsupported torchvision architecture: {arch}")


def build_model() -> nn.Module:
    if _is_swin(ARCHITECTURE):
        assert TIMM_AVAILABLE, "Install timm:  pip install timm>=0.9.0"
        backbone = timm.create_model(
            ARCHITECTURE,
            pretrained  = bool(PRETRAINED),
            num_classes = 0,       # remove timm head — we supply our own
            global_pool = "avg",
        )
        feat_dim = backbone.num_features   # 768 for SwinV2-Small

        # Custom classification head
        backbone.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(p=float(DROPOUT)),
            nn.Linear(feat_dim, int(NUM_CLASSES)),
        )

        # Patch forward so backbone.head is called after feature extraction
        _orig_fwd = backbone.forward_features
        def _new_forward(x):
            return backbone.head(_orig_fwd(x))
        backbone.forward = _new_forward

        total  = sum(p.numel() for p in backbone.parameters()) / 1e6
        print(f"SwinV2-Small | {total:.1f}M params | feat_dim={feat_dim}")
        print(f"Pretrained: {PRETRAINED}  |  Head dropout: {DROPOUT}")
        return backbone

    else:
        # Fallback: torchvision EfficientNet / ViT (for teammates)
        if not hasattr(tv_models, ARCHITECTURE):
            raise ValueError(f"Unknown torchvision model: {ARCHITECTURE}")
        m = getattr(tv_models, ARCHITECTURE)(
            weights="DEFAULT" if bool(PRETRAINED) else None)
        replace_classifier_head(m, ARCHITECTURE, int(NUM_CLASSES), float(DROPOUT))
        return m


def set_backbone_trainable(model: nn.Module, architecture: str,
                           train_backbone: bool) -> None:
    """Freeze/unfreeze backbone. Head always stays trainable."""
    if _is_swin(architecture):
        for name, p in model.named_parameters():
            p.requires_grad = True if "head" in name else bool(train_backbone)
    elif architecture.lower().startswith("vit"):
        for name, p in model.named_parameters():
            p.requires_grad = bool(train_backbone) if not name.startswith("heads") else True
    elif architecture.lower().startswith("efficientnet"):
        for name, p in model.named_parameters():
            p.requires_grad = bool(train_backbone) if name.startswith("features") else True
    else:
        for p in model.parameters():
            p.requires_grad = True


model = build_model().to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Trainable params (before freeze): {trainable:.1f}M")

## 7) Loss, optimiser, scheduler

In [ ]:
# 7) Loss + optimiser + scheduler

def compute_class_weights(y: np.ndarray, num_classes: int) -> torch.Tensor:
    """Inverse-frequency class weights for CrossEntropyLoss."""
    counts  = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    weights = (len(y) / (num_classes * counts)).astype(np.float32)
    return torch.tensor(weights, device=device)


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).
    Down-weights easy examples (the many grade-0 images the model
    already classifies correctly) and focuses on hard minority cases.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma: float = 2.0,
                 alpha: float | None = None,
                 weight: torch.Tensor | None = None):
        super().__init__()
        self.gamma  = float(gamma)
        self.alpha  = None if alpha is None else float(alpha)
        self.weight = weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce   = F.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            loss = self.alpha * loss
        return loss.mean()


# ── Build loss ────────────────────────────────────────────────────────────────
set_backbone_trainable(model, ARCHITECTURE,
                       train_backbone=(int(FREEZE_BACKBONE_EPOCHS) == 0))

class_weights = None
if CLASS_WEIGHTS_AUTO:
    class_weights = compute_class_weights(train_df["level"].to_numpy(), int(NUM_CLASSES))
    print("Class weights:", class_weights.cpu().numpy().round(3))

if USE_FOCAL_LOSS:
    criterion = FocalLoss(gamma=float(FOCAL_GAMMA), alpha=FOCAL_ALPHA,
                          weight=class_weights)
    print(f"Loss: FocalLoss (gamma={FOCAL_GAMMA})")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights,
                                    label_smoothing=float(LABEL_SMOOTHING))
    print(f"Loss: CrossEntropyLoss (label_smoothing={LABEL_SMOOTHING})")

# ── Layer-wise LR: backbone gets 10x lower LR than head ───────────────────────
# Prevents catastrophic forgetting of ImageNet-22k pre-trained representations
if _is_swin(ARCHITECTURE):
    head_params     = [p for n, p in model.named_parameters()
                       if "head" in n and p.requires_grad]
    backbone_params = [p for n, p in model.named_parameters()
                       if "head" not in n and p.requires_grad]
    param_groups = [
        {"params": backbone_params, "lr": float(LR) * 0.1},
        {"params": head_params,     "lr": float(LR)},
    ]
else:
    param_groups = [p for p in model.parameters() if p.requires_grad]

if OPTIMIZER.lower() == "adamw":
    optimizer = torch.optim.AdamW(param_groups, lr=float(LR),
                                  weight_decay=float(WEIGHT_DECAY))
elif OPTIMIZER.lower() == "sgd":
    optimizer = torch.optim.SGD(param_groups, lr=float(LR),
                                momentum=0.9, weight_decay=float(WEIGHT_DECAY))
else:
    raise ValueError(f"Unknown optimizer: {OPTIMIZER}")

scheduler = None
if SCHEDULER == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=int(EPOCHS), eta_min=1e-7)
elif SCHEDULER == "step":
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=int(SCHEDULER_STEP_SIZE), gamma=float(SCHEDULER_GAMMA))

print(f"Optimiser: {OPTIMIZER}  |  Scheduler: {SCHEDULER}")

## 8) Training loop

Metrics tracked: quadratic-weighted Kappa, macro F1, loss.
Accuracy is intentionally **not** used — class imbalance makes it misleading.

In [ ]:
# 8) Training loop — AMP, warm-up freeze, MixUp, early stopping, QWK

use_amp     = bool(MIXED_PRECISION) and device.type == "cuda"
scaler      = torch.cuda.amp.GradScaler() if use_amp else None
use_mixup   = AUGMENT_VERSION.lower() == "v4_mixup"

def run_epoch(loader, train_mode: bool):
    """Single training or validation pass. Returns (loss, kappa, y_true, y_pred)."""
    model.train(train_mode)
    total_loss  = 0.0
    total_n     = 0
    preds_list  = []
    labels_list = []

    for images, labels in tqdm(loader, leave=False,
                               desc="train" if train_mode else "val"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        # MixUp — only during training, only for v4_mixup
        mixed_labels_b, lam = None, 1.0
        if train_mode and use_mixup:
            images, labels, mixed_labels_b, lam = mixup_batch(
                images, labels, alpha=float(MIXUP_ALPHA))

        ctx = (torch.autocast(device_type="cuda", enabled=True)
               if use_amp else nullcontext())
        with ctx:
            logits = model(images)
            if train_mode and use_mixup and mixed_labels_b is not None:
                loss = (lam * criterion(logits, labels)
                        + (1 - lam) * criterion(logits, mixed_labels_b))
            else:
                loss = criterion(logits, labels)

        if train_mode:
            if use_amp and scaler is not None:
                scaler.scale(loss).backward()
                if GRADIENT_CLIP_NORM:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(),
                                             float(GRADIENT_CLIP_NORM))
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                if GRADIENT_CLIP_NORM:
                    nn.utils.clip_grad_norm_(model.parameters(),
                                             float(GRADIENT_CLIP_NORM))
                optimizer.step()

        total_loss += float(loss.item()) * labels.size(0)
        total_n    += labels.size(0)
        preds_list.append(logits.argmax(1).detach().cpu())
        labels_list.append((labels if mixed_labels_b is None
                             else labels).detach().cpu())

    y_pred  = torch.cat(preds_list).numpy()
    y_true  = torch.cat(labels_list).numpy()
    avg_loss= total_loss / max(total_n, 1)
    kappa   = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    return avg_loss, kappa, y_true, y_pred


# ── Checkpoint dir ────────────────────────────────────────────────────────────
ckpt_dir = (REPO_ROOT / "checkpoints").resolve()
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_path = ckpt_dir / f"{EXPERIMENT_NAME}_best.pt"
log_path  = ckpt_dir / f"{EXPERIMENT_NAME}_log.json"

freeze_epochs    = int(FREEZE_BACKBONE_EPOCHS)
best_metric_val  = -1.0
patience_left    = int(EARLY_STOPPING_PATIENCE)
history          = {"train_loss": [], "val_loss": [],
                    "val_kappa": [], "epoch": []}

print(f"\nTraining {EXPERIMENT_NAME}")
print(f"Augment: {AUGMENT_VERSION}  |  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}")
print(f"Freeze backbone for first {freeze_epochs} epoch(s), then full fine-tune")
print("-" * 70)

for epoch in range(int(EPOCHS)):
    # Warm-up: unfreeze backbone after freeze_epochs
    train_backbone = (epoch >= freeze_epochs)
    set_backbone_trainable(model, ARCHITECTURE, train_backbone=train_backbone)

    # Rebuild optimiser when warm-up ends so unfrozen params are included
    if train_backbone and epoch == freeze_epochs and freeze_epochs > 0:
        if _is_swin(ARCHITECTURE):
            head_p = [p for n, p in model.named_parameters()
                      if "head" in n and p.requires_grad]
            back_p = [p for n, p in model.named_parameters()
                      if "head" not in n and p.requires_grad]
            pg = [{"params": back_p, "lr": float(LR) * 0.1},
                  {"params": head_p, "lr": float(LR)}]
        else:
            pg = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(pg, lr=float(LR),
                                      weight_decay=float(WEIGHT_DECAY))
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, int(EPOCHS) - epoch), eta_min=1e-7)
        print(f"  [Epoch {epoch+1}] Backbone unfrozen — full fine-tune starts")

    train_loss, train_kappa, _, _  = run_epoch(train_loader, train_mode=True)
    val_loss,   val_kappa,   _, _  = run_epoch(val_loader,   train_mode=False)
    if scheduler is not None:
        scheduler.step()

    cur_lr = optimizer.param_groups[-1]["lr"]
    history["train_loss"].append(round(train_loss, 5))
    history["val_loss"].append(round(val_loss, 5))
    history["val_kappa"].append(round(val_kappa, 5))
    history["epoch"].append(epoch + 1)

    improved = val_kappa > best_metric_val and not np.isnan(val_kappa)
    mark = "★" if improved else " "
    print(f"Ep {epoch+1:3d}/{EPOCHS} {mark}| "
          f"train {train_loss:.4f} κ={train_kappa:.4f} | "
          f"val {val_loss:.4f} κ={val_kappa:.4f} | lr {cur_lr:.2e}")

    if improved:
        best_metric_val = val_kappa
        patience_left   = int(EARLY_STOPPING_PATIENCE)
        torch.save({
            "model_state" : model.state_dict(),
            "epoch"       : epoch,
            "val_kappa"   : val_kappa,
            "val_loss"    : val_loss,
            "config"      : {
                "EXPERIMENT_NAME"  : EXPERIMENT_NAME,
                "ARCHITECTURE"     : ARCHITECTURE,
                "AUGMENT_VERSION"  : AUGMENT_VERSION,
                "WEIGHTED_SAMPLING": WEIGHTED_SAMPLING,
                "USE_FOCAL_LOSS"   : USE_FOCAL_LOSS,
                "CLASS_WEIGHTS_AUTO": CLASS_WEIGHTS_AUTO,
                "IMAGE_SIZE"       : IMAGE_SIZE,
                "BATCH_SIZE"       : BATCH_SIZE,
                "LR"               : LR,
            },
        }, best_path)
        print(f"  Saved best (κ={val_kappa:.4f}) → {best_path}")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print("Early stopping triggered.")
            break

# Save training log (upload to GitHub with commit)
with open(log_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"\nTraining complete. Best val κ = {best_metric_val:.4f}")
print(f"Log saved → {log_path}")  

## 9) Evaluation — metrics, confusion matrix, TTA

**Metrics used** (accuracy is excluded — class imbalance makes it misleading):
- Quadratic-weighted Kappa (standard for ordinal medical grading)
- Macro F1, Precision, Recall (treat all classes equally)
- ROC-AUC one-vs-rest (discrimination ability)
- Per-class F1 (shows which DR grade is hardest)

In [ ]:
# 9) Load best checkpoint and evaluate — with optional TTA

checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']+1}  "
      f"(val κ={checkpoint['val_kappa']:.4f})")


# ── Standard evaluation ───────────────────────────────────────────────────────
_, std_kappa, y_true, y_pred = run_epoch(val_loader, train_mode=False)

labels_names = ["0-No DR", "1-Mild", "2-Moderate", "3-Severe", "4-Proliferative"]

print("\n" + "="*60)
print(f"  {EXPERIMENT_NAME} — Standard Evaluation")
print("="*60)
print(f"  Quadratic Kappa : {std_kappa:.4f}")
print(f"  Macro F1        : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"  Macro Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"  Macro Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print()
print("  Per-class F1:")
per_cls = f1_score(y_true, y_pred, average=None, zero_division=0)
for name, f1 in zip(labels_names, per_cls):
    bar = "█" * int(f1 * 20)
    print(f"    {name:22s}: {f1:.4f}  {bar}")
print()
print(classification_report(y_true, y_pred, target_names=labels_names))


# ── Test-time augmentation (TTA) ──────────────────────────────────────────────
def predict_with_tta(model: nn.Module, loader: DataLoader, n: int = 5):
    """
    Test-time augmentation — average softmax over N augmented views.
    Views: original + h-flip + v-flip + rot90 + rot270
    Typically adds +2-4% macro F1 with zero additional training.
    """
    tta_transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),           # horizontal flip
        lambda x: torch.flip(x, dims=[2]),           # vertical flip
        lambda x: torch.rot90(x, k=1, dims=[2, 3]),  # 90°
        lambda x: torch.rot90(x, k=3, dims=[2, 3]),  # 270°
    ][:n]

    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"TTA ({n} views)"):
            images = images.to(device)
            probs  = torch.stack([
                F.softmax(model(tfm(images)), dim=1)
                for tfm in tta_transforms
            ]).mean(0)   # average over N views
            all_probs.append(probs.cpu())
            all_preds.append(probs.argmax(1).cpu())
            all_labels.append(labels)

    y_pred_tta  = torch.cat(all_preds).numpy()
    y_true_tta  = torch.cat(all_labels).numpy()
    probs_all   = torch.cat(all_probs).numpy()
    kappa_tta   = cohen_kappa_score(y_true_tta, y_pred_tta, weights="quadratic")
    macro_f1    = f1_score(y_true_tta, y_pred_tta, average="macro", zero_division=0)
    return y_true_tta, y_pred_tta, probs_all, kappa_tta, macro_f1


if USE_TTA:
    y_true_tta, y_pred_tta, probs_tta, kappa_tta, f1_tta = predict_with_tta(
        model, val_loader, n=TTA_N)
    print(f"\nTTA ({TTA_N} views)  |  κ={kappa_tta:.4f}  |  macro F1={f1_tta:.4f}")
    print(f"Δ kappa vs standard: {kappa_tta - std_kappa:+.4f}")
    print()
    print(classification_report(y_true_tta, y_pred_tta, target_names=labels_names))

In [ ]:
# 10) Confusion matrix — standard and normalised

out_dir = (REPO_ROOT / "outputs").resolve()
out_dir.mkdir(parents=True, exist_ok=True)

use_preds  = y_pred_tta  if (USE_TTA and "y_pred_tta" in dir()) else y_pred
use_labels = y_true_tta  if (USE_TTA and "y_true_tta" in dir()) else y_true
suffix     = "_tta" if USE_TTA else ""

cm      = confusion_matrix(use_labels, use_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
short_names = ["No DR", "Mild", "Moderate", "Severe", "Prolif."]

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    [f"{EXPERIMENT_NAME} — counts", f"{EXPERIMENT_NAME} — normalised"],
    ["d", ".2f"],
):
    im = ax.imshow(data, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(5), yticks=np.arange(5),
           xticklabels=short_names, yticklabels=short_names,
           xlabel="Predicted", ylabel="True", title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, format(data[i, j], fmt),
                    ha="center", va="center",
                    color="white" if data[i, j] > data.max() * 0.5 else "black",
                    fontsize=9)

fig.tight_layout()
fig_path = out_dir / f"{EXPERIMENT_NAME}_confusion{suffix}.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Saved confusion matrix →", fig_path)
plt.show()

In [ ]:
# 11) Training curves — loss and kappa over epochs

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="Train loss", color="#2563EB")
axes[0].plot(history["epoch"], history["val_loss"],   label="Val loss",   color="#DC2626")
axes[0].set_title(f"{EXPERIMENT_NAME} — Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
# Mark warm-up boundary
if int(FREEZE_BACKBONE_EPOCHS) > 0:
    axes[0].axvline(x=int(FREEZE_BACKBONE_EPOCHS), color="gray",
                    linestyle="--", alpha=0.7, label="Unfreeze")

axes[1].plot(history["epoch"], history["val_kappa"], label="Val κ (QWK)",
             color="#16A34A", linewidth=2)
axes[1].axhline(y=best_metric_val, color="orange", linestyle="--",
                alpha=0.8, label=f"Best κ={best_metric_val:.4f}")
axes[1].set_title(f"{EXPERIMENT_NAME} — Quadratic Kappa")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Kappa")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
curves_path = out_dir / f"{EXPERIMENT_NAME}_curves.png"
fig.savefig(curves_path, dpi=150, bbox_inches="tight")
print("Saved training curves →", curves_path)
plt.show()

## 12) Grad-CAM — retinal attention visualisation

Shows *where* the model is looking when making its prediction.
Used to verify the model attends to clinically relevant regions
(optic disc, macula, vessel abnormalities) rather than image artefacts.

In [ ]:
# 12) Grad-CAM — works for SwinV2 and EfficientNet

class GradCAM:
    """
    Gradient-weighted Class Activation Mapping.

    For SwinV2: target the LayerNorm after the last Swin stage.
    The output is a sequence of tokens → we reshape to spatial grid.
    """
    def __init__(self, mdl: nn.Module, target_layer: nn.Module):
        self.mdl          = mdl
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        target_layer.register_forward_hook(
            lambda m, inp, out: setattr(self, "activations", out.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, "gradients", go[0].detach()))

    def __call__(self, x: torch.Tensor, class_idx: int | None = None):
        self.mdl.eval()
        x      = x.unsqueeze(0).to(device)
        logits = self.mdl(x)
        if class_idx is None:
            class_idx = int(logits.argmax(1).item())
        self.mdl.zero_grad()
        logits[0, class_idx].backward()

        grads = self.gradients    # (1, N, C) for Swin token output
        acts  = self.activations

        # Swin outputs token sequences — reshape to spatial grid
        if grads.ndim == 3:
            B, N, C   = grads.shape
            gs        = int(N ** 0.5)
            grads = grads.reshape(B, gs, gs, C).permute(0, 3, 1, 2)
            acts  = acts.reshape(B, gs, gs, C).permute(0, 3, 1, 2)

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * acts).sum(dim=1).squeeze(0))
        cam     = cam.cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam     = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))
        return cam, class_idx


def show_gradcam(dataset: Dataset, indices: list, target_layer: nn.Module,
                 model_name: str = "SwinV2-Small"):
    grad_cam = GradCAM(model, target_layer)
    grades   = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
    n        = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[None]

    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)

    for row, idx in enumerate(indices):
        img_t, true_lbl = dataset[idx]
        cam, pred_lbl   = grad_cam(img_t)

        orig     = img_t.permute(1, 2, 0).numpy()
        orig     = np.clip(orig * std + mean, 0, 1)
        orig_u8  = (orig * 255).astype(np.uint8)
        heat     = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heat     = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
        overlay  = (0.5 * heat + 0.5 * orig_u8).astype(np.uint8)

        for col, (im, title) in enumerate([
            (orig,    f"True: {grades[true_lbl]}"),
            (cam,     f"Grad-CAM | Pred: {grades[pred_lbl]}"),
            (overlay, f"{model_name} overlay"),
        ]):
            axes[row, col].imshow(im, cmap="jet" if col == 1 else None)
            axes[row, col].set_title(title, fontsize=10)
            axes[row, col].axis("off")

    plt.suptitle(f"{model_name} — Grad-CAM Visualisations", fontsize=13)
    plt.tight_layout()
    cam_path = out_dir / f"{EXPERIMENT_NAME}_gradcam.png"
    plt.savefig(cam_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved Grad-CAM →", cam_path)


# Target: LayerNorm after last Swin stage's last block
# This is where the highest-level spatial features are encoded
try:
    if _is_swin(ARCHITECTURE):
        # SwinV2-Small: layers[-1] is the last stage
        target = model.layers[-1].blocks[-1].norm1
    else:
        # EfficientNet fallback
        target = model.features[-1]

    # Pick one sample from each DR grade for the visualisation
    grade_indices = []
    for grade in range(NUM_CLASSES):
        grade_rows = val_df[val_df["level"] == grade]
        if len(grade_rows) > 0:
            grade_indices.append(grade_rows.index[0] % len(val_ds))

    show_gradcam(val_ds, grade_indices[:5], target)

except Exception as e:
    print(f"Grad-CAM failed: {e}")
    print("This can happen if the model forward was patched — "
          "try running after model is loaded from checkpoint.")

In [ ]:
# 13) Save experiment summary — commit this to GitHub

summary = {
    "experiment"        : EXPERIMENT_NAME,
    "architecture"      : ARCHITECTURE,
    "augment_version"   : AUGMENT_VERSION,
    "imbalance_strategy": (
        "focal_loss"       if USE_FOCAL_LOSS       else
        "weighted_sampler" if WEIGHTED_SAMPLING     else
        "class_weights"    if CLASS_WEIGHTS_AUTO    else
        "none"
    ),
    "pretrained"        : PRETRAINED,
    "image_size"        : IMAGE_SIZE,
    "batch_size"        : BATCH_SIZE,
    "lr"                : LR,
    "epochs_run"        : len(history["epoch"]),
    "best_val_kappa"    : round(best_metric_val, 5),
    "best_val_f1_macro" : round(f1_score(y_true, y_pred,
                                          average="macro",
                                          zero_division=0), 5),
    "tta_kappa"         : round(kappa_tta, 5) if USE_TTA else None,
    "tta_f1_macro"      : round(f1_tta, 5)    if USE_TTA else None,
    "history"           : history,
}

summary_path = out_dir / f"{EXPERIMENT_NAME}_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print(f"  Experiment: {EXPERIMENT_NAME}")
print(f"  Architecture: {ARCHITECTURE}")
print(f"  Augmentation: {AUGMENT_VERSION}")
print(f"  Imbalance:    {summary['imbalance_strategy']}")
print(f"  Best val κ:   {best_metric_val:.4f}")
print(f"  Macro F1:     {summary['best_val_f1_macro']:.4f}")
if USE_TTA:
    print(f"  TTA κ:        {kappa_tta:.4f}  (Δ={kappa_tta-best_metric_val:+.4f})")
print(f"\nSummary saved → {summary_path}")
print("Upload outputs/ and checkpoints/*_log.json to GitHub with a")
print("descriptive commit: e.g. 'Swin V3 full augment + weighted sampler — κ=0.XX'")